# Workshop 10. A practice case: choosing what to model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture10/workshop10.ipynb)

This workshop is practice in using regression on a question that comes without a model. You get management's
questions and the data, and you decide which column to model, which explanatory variables belong in the model, and
which of two similar models to keep. Work in your group or alone, and ask the TAs when you are stuck.

There is often more than one defensible answer. What counts is that you can say why you chose as you did, and what in
the output supports it. Write your reasons in the answer cells; they are the practice for explaining your choices
without notes.

Each task has two hints: the first names the statistical step, the second names the functions. Try the task before
you open them. The toolbox says where you have seen each function before. The slides, the lecture notebook and the
notebooks of Lectures 6 to 9 are your main references. You are welcome to use an LLM, but every task can be solved
without one.

A few rules from the lectures that apply throughout: compare two models on the same rows; use robust (HC1) standard
errors in linear regression; in logit, read the sign and significance from the coefficients and the sizes as
probabilities or percentage points.

## The case

TeleNag sells internet and mobile subscriptions. Its customer service center handles contacts by phone and chat.
Some agents have taken a course on handling and documenting contacts; managers chose which agents to send. The course
has been offered during the past year. The file `service_contacts.csv` has one row for each of 2,000 contacts drawn
at random from all contacts this autumn. The center has several sites and many hundreds of agents, so most agents
appear only once or twice in the file. The company is invented and the data are
simulated for the course.

Management asks two questions:

- **A. Agent time.** What goes with how much agent time a contact takes, and is the course associated with less agent
  time?
- **B. Repeat contacts.** Which contacts are followed by the customer contacting the center again within seven days,
  and is the course associated with fewer of them?

| Column | What it measures |
|---|---|
| `contact_id` | Running number of the contact |
| `channel` | "Phone" or "Chat" |
| `issue_type` | What the customer contacted the center about: "Billing", "Technical", "Plan change" or "Cancellation" |
| `segment` | "Private" or "Business" customer |
| `shift` | When the contact came in: "Morning", "Afternoon" or "Evening" |
| `customer_years` | Years the customer has been with TeleNag |
| `agent_experience_months` | The agent's months in the job |
| `agent_contacts_handled` | The number of contacts the agent has handled in the job so far |
| `agent_trained` | 1 if the agent has taken the course |
| `agent_minutes_before` | The agent's average time per contact (`total_minutes`) in the same autumn period one year earlier, before the course started; empty for agents who were not in the job then |
| `queue_minutes` | Minutes the customer waited before the contact was answered |
| `transfers` | Times the contact was passed on to another agent while it went on |
| `escalated` | 1 if the agent passed the issue on to the second-line team at the end of the contact |
| `handling_minutes` | Minutes the agent spent talking or chatting with the customer, including time on hold |
| `wrapup_minutes` | Minutes the agent spent on the contact after the customer had left (notes and follow-up tasks) |
| `total_minutes` | `handling_minutes` plus `wrapup_minutes` |
| `repeat_contact_7d` | 1 if the customer contacted the center again within seven days |
| `satisfaction` | 1 (very dissatisfied) to 5 (very satisfied), from a survey sent the day after the contact; empty if the customer did not answer |

## Toolbox

The functions you need today, and where you have seen them in use.

| Function | What it does | Seen in |
|---|---|---|
| `df.describe()`, `df["x"].skew()`, `plt.hist(df["x"], bins=40)` | Summaries, skewness and a histogram | Lecture 5; Lecture 7, Part 3 |
| `df.corr(numeric_only=True)` | Correlations between the numeric columns | Lecture 6 |
| `smf.ols("np.log(y) ~ x1 + C(g)", data=df).fit(cov_type="HC1")` | Linear regression on log y with robust standard errors | Lecture 7 |
| `model.params`, `model.pvalues`, `model.conf_int()`, `model.rsquared_adj` | Coefficients, p-values, intervals, adjusted R² | Lectures 6 and 7 |
| `np.exp(b) - 1` | The exact percent difference for a coefficient b on log y | Lecture 7, Part 3 |
| `plt.scatter(model.fittedvalues, model.resid)` | Residuals against fitted values | Lecture 7, Part 1 |
| `het_breuschpagan(model.resid, model.model.exog)` | Breusch-Pagan test; the second number is the p-value | Lecture 7, Part 1 |
| `variance_inflation_factor(model.model.exog, i)` | VIF for column i (leave out i = 0, the intercept) | Lecture 7, Part 4 |
| `smf.logit("y01 ~ x1 + C(g)", data=df).fit()` | A logit model | Lecture 9, Part 2 |
| `model.llf`, `model.prsquared`, `stats.chi2.sf(LR, q)` | Log-likelihood, pseudo R², and the p-value of a likelihood ratio test | Lecture 9, Part 2 |
| `model.get_margeff(at="overall", dummy=True).summary()` | Average marginal effects | Lecture 9, Part 4 |
| `model.predict(pd.DataFrame({...}))` | Predictions for cases you describe | Lecture 9, Part 3; Lecture 10, Part 4 |
| `df.dropna(subset=["x"])` | The rows where x is not missing | Lecture 5 |

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
contacts = pd.read_csv(url + "service_contacts.csv")
contacts.head()

## Part A. Agent time

**Management's question.** What goes with how much agent time a contact takes, and is the course associated with
less agent time?

### Task 1. Which column is y?

Three columns measure time spent by the agent in some way, and one measures time spent by the customer. Look at them.
Which one answers management's question, and should it be modeled in levels or in logs? Give your reasons.

<details>
<summary>Hint 1</summary>

Which of the columns are time the agent spends on the contact, and when? For the form, look at the shape of the
distribution in levels and in logs.
</details>

<details>
<summary>Hint 2</summary>

`contacts[["handling_minutes", "wrapup_minutes", "total_minutes", "queue_minutes"]].describe()`, `.skew()`, and
`plt.hist(np.log(contacts["total_minutes"]), bins=40)`.
</details>

**Your answer:**

### Task 2. Which columns are candidates for the x's?

Go through the other columns with the questions from the lecture: Is it known before the outcome happens? Is it a part
of the outcome? Is it nearly the same as another column? Is it only a label? Does the question need it? Sort them into
candidates and columns that do not belong, and say why for each. You may look at the data while you decide.

<details>
<summary>Hint 1</summary>

Think about when each column is known compared with the agent time it should explain: before the contact starts,
during it, or after it.
</details>

<details>
<summary>Hint 2</summary>

The column descriptions above, and `contacts.corr(numeric_only=True).round(2)` to spot near copies.
</details>

**Your answer:**

### Task 3. Two similar models

At each decision point below, fit both models on the same rows, read both outputs, and decide which one to keep.
Include the controls you consider necessary from Task 2 in both models, so the two differ only in what is named.

**(a) The course with and without the agent's experience.** What happens to the coefficient on `agent_trained`, and
why?

<details>
<summary>Hint 1</summary>

If the coefficient moves a lot when experience is added, look at how experience differs between agents with and
without the course. This is omitted variable bias (Lecture 7).
</details>

<details>
<summary>Hint 2</summary>

`smf.ols(...).fit(cov_type="HC1")` twice; `contacts.groupby("agent_trained")["agent_experience_months"].mean()`.
</details>

**Your answer:**

**(b) How to put the agent's experience in the model.** Compare experience in months with the log of experience. Then
try `agent_contacts_handled` in place of experience, and both together. Which do you keep, and why?

<details>
<summary>Hint 1</summary>

Ask whether one more month should matter as much for an agent in month 2 as for one in month 100. For the pair, look
at the standard errors and the VIF when both are in.
</details>

<details>
<summary>Hint 2</summary>

`np.log(agent_experience_months)` in the formula; `model.bse`; `variance_inflation_factor(model.model.exog, i)`.
</details>

**Your answer:**

**(c) With and without `transfers`.** Does it belong in a model for management's question?

<details>
<summary>Hint 1</summary>

Ask when a transfer happens, and whether it is something that explains agent time or a part of how the time arises.
Look at what happens to the experience coefficient.
</details>

<details>
<summary>Hint 2</summary>

Fit your model from (b) with and without `+ transfers` and compare `params`, `pvalues` and `rsquared_adj`.
</details>

**Your answer:**

**(d) With and without the agent's earlier time per contact.** `agent_minutes_before` is the agent's average time per
contact in the same period one year earlier, before the course started. Fit your model with and without it, on the
same rows. What happens to the course coefficient, and what does it measure once the earlier value is in the model? Which
contacts does the model now describe?

<details>
<summary>Hint 1</summary>

Drop the rows with a missing earlier value before you fit either model, so the two use the same rows. Then ask: what
do agents with the same earlier value have in common? (Lecture 7, and the slide on an earlier value in Lecture 10.)
</details>

<details>
<summary>Hint 2</summary>

`with_before = contacts.dropna(subset=["agent_minutes_before"])`; `np.log(agent_minutes_before)` in the formula;
`with_before.groupby("agent_trained")["agent_minutes_before"].mean()`.
</details>

**Your answer:**

### Task 4. Your model

Fit the model you have chosen with robust standard errors. Decide about `queue_minutes`, `shift` and `customer_years`.
Check the model (residuals against fitted values, Breusch-Pagan, VIF), and read the output: what goes with more and
with less agent time, and what does the model say about the course, in percent? Say which contacts the model
describes.

<details>
<summary>Hint 1</summary>

For a group of x's, compare adjusted R² and look at their p-values together, on the same rows. Read a coefficient b on
log y as about 100·b percent, exactly 100·(e^b − 1) percent.
</details>

<details>
<summary>Hint 2</summary>

`model.summary()`, `plt.scatter(model.fittedvalues, model.resid)`, `het_breuschpagan`, `variance_inflation_factor`,
`np.exp(b) - 1`, `model.conf_int()`.
</details>

**Your answer:**

## Part B. Repeat contacts

**Management's question.** Which contacts are followed by the customer contacting the center again within seven days,
and is the course associated with fewer of them?

### Task 5. y and the candidates

Which column is y, and which kind of model fits it? Which columns from Task 2 are candidates here, and are there any
that change status because the outcome is different? Give your reasons.

<details>
<summary>Hint 1</summary>

Ask the same questions as in Task 2, now about a repeat contact within seven days. When is it known compared with
`escalated`, `transfers` and `satisfaction`?
</details>

<details>
<summary>Hint 2</summary>

`contacts["repeat_contact_7d"].mean()`, `contacts["satisfaction"].isna().mean()`,
`contacts.groupby("escalated")["repeat_contact_7d"].mean()`.
</details>

**Your answer:**

### Task 6. Two similar models

Start from the x's you chose for Part A (known at the start), now in a logit model for `repeat_contact_7d`.

**(a) With and without `satisfaction`.** What happens to the number of rows, to the course coefficient and to the
pseudo R²? Fit the two models on the same rows as well. What do you conclude?

<details>
<summary>Hint 1</summary>

A model with `satisfaction` can only use the rows where it is answered. Compare the course coefficient on all rows,
on the answered rows without `satisfaction`, and on the answered rows with it.
</details>

<details>
<summary>Hint 2</summary>

`answered = contacts.dropna(subset=["satisfaction"])`; `smf.logit(..., data=answered).fit()`; `model.nobs`.
</details>

**Your answer:**

**(b) With and without `escalated`.** Does it belong? Does your answer depend on what the model is for?

<details>
<summary>Hint 1</summary>

Think of two purposes: deciding at the end of a contact whom to call back, and understanding what goes with repeat
contacts, including the course.
</details>

<details>
<summary>Hint 2</summary>

Fit your model with and without `+ escalated`; compare the course coefficient, the pseudo R², and the likelihood ratio
test `2 * (m2.llf - m1.llf)`.
</details>

**Your answer:**

**(c) The remaining candidates.** Are `queue_minutes`, `shift` and `customer_years` worth adding? Use a likelihood
ratio test for the group.

<details>
<summary>Hint 1</summary>

A likelihood ratio test compares the log-likelihoods of the model with and without a group of x's; the degrees of
freedom are the number of coefficients added.
</details>

<details>
<summary>Hint 2</summary>

`LR = 2 * (large.llf - small.llf)`, `q = len(large.params) - len(small.params)`, `stats.chi2.sf(LR, q)`.
</details>

**Your answer:**

### Task 7. Your model

Fit the logit model you have chosen and report the average marginal effects. Read the output: which contacts are more
and less often followed by a repeat contact, and what does the model say about the course, in percentage points?
Check that the LPM with the same x's gives the same conclusions.

<details>
<summary>Hint 1</summary>

The AMEs give the sizes in percentage points; `dummy=True` gives the change from 0 to 1 for a yes/no variable.
</details>

<details>
<summary>Hint 2</summary>

`model.get_margeff(at="overall", dummy=True).summary()`; `smf.ols(formula, data=contacts).fit(cov_type="HC1")` for the
LPM; `(lpm.fittedvalues < 0).sum()`.
</details>

**Your answer:**

## Extension

**Repeat contacts for agents with and without the course.** For a private customer who calls by phone about a
technical issue,
make a grid of predicted repeat contact for agents with 3, 12, 36 and 96 months in the job, with and without the
course. Where is the difference between agents with and without the course largest, and why?

**Your answer:**